# Ingest and Govern Logistics Data with Open Data Lakehouse
### *Lakehouse Apache Iceberg REST Catalog, Managed Service for Apache Spark, BigQuery, and Knowledge Catalog Data Insights*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GoogleCloudPlatform/devrel-demos/blob/main/data-analytics/data-cloud-roadshow/lab1/lab1_ingest_and_govern_logistics_data.ipynb)

---

## Architecture & Workflow Overview
In this notebook, you will build and govern an enterprise **Open Data Lakehouse** on Google Cloud using open standards (**Apache Iceberg**) and native services (**Lakehouse**, **Managed Service for Apache Spark**, **BigQuery**, and **Knowledge Catalog**).


### Key Objectives
1. **Initialize Cloud Environment**: Authenticate, set project variables, and enable required Google Cloud APIs.
2. **Provision Lakehouse Iceberg REST Catalog**: Create a Cloud Storage bucket, provision a **Lakehouse Iceberg REST Catalog**, and define a metastore **namespace**.
3. **Ingest & Transform with Serverless PySpark**: Submit a **Managed Service for Apache Spark serverles** job that reads from a Cloud Storage bucket, cleans and enriches the records, and writes directly into the Lakehouse Apache Iceberg table.
4. **Query Lakehouse from BigQuery**: Query the Apache Iceberg table seamlessly using standard BigQuery SQL.
5. **Generate Automated Metadata with Knowledge Catalog**: Use **Knowledge Catalog Data Insights** and Gemini to generate natural language table summaries, column descriptions, and data queries.
6. **Clean Up Resources**: Optional teardown script to remove provisioned assets.



---
## 1. Environment Setup & Dependencies

Install the required dependencies.



In [ ]:
# Install required libraries
!pip install --quiet --upgrade \
    dataproc-spark-connect \
    google-cloud-bigquery \
    google-cloud-storage \
    google-cloud-bigquery-storage \
    pyarrow

print("Required packages installed successfully.")


### Configure Project & Resource Parameters
Set your `PROJECT_ID`, `REGION` (e.g. `us-central1`), and lakehouse naming parameters.



In [ ]:
import os
import sys
from google.cloud import bigquery
from google.cloud import storage

# =====================================================================
# CONFIGURATION - UPDATE THESE VALUES AS NEEDED
# =====================================================================
PROJECT_ID = "YOUR_PROJECT_ID"  # <-- Replace with your GCP project ID
REGION = os.environ.get("REGION", "YOUR_REGION") # <-- Replace with a region, i.e. 'us-central1'
LAKEHOUSE_BUCKET_NAME = f"{PROJECT_ID}-lakehouse-bucket"
CATALOG_NAME = LAKEHOUSE_BUCKET_NAME
NAMESPACE_ID = "taxi_namespace"
TABLE_NAME = "processed_taxi_data"

# For setup
EXTERNAL_BUCKET_NAME = f"{PROJECT_ID}-external-bucket"

print("====================================================")
print(f"Project ID:             {PROJECT_ID}")
print(f"Region:                 {REGION}")
print(f"Warehouse GCS Bucket:   gs://{LAKEHOUSE_BUCKET_NAME}")
print(f"Iceberg Catalog:        {CATALOG_NAME} ({LAKEHOUSE_BUCKET_NAME})")
print(f"Iceberg Namespace:      {NAMESPACE_ID}")
print(f"Iceberg Table:          {TABLE_NAME}")
print("====================================================")



### Enable Required Google Cloud APIs
Enable the necessary APIs for BigQuery, Cloud Storage, Lakehouse, Knowledge Catalog, and Managed Service for Apache Spark.



In [ ]:
import subprocess

# Enable required GCP APIs
subprocess.run([
    "gcloud", "services", "enable", 
    "bigquery.googleapis.com",
    "biglake.googleapis.com",
    "storage.googleapis.com",
    "dataplex.googleapis.com",
    "datacatalog.googleapis.com",
    "dataproc.googleapis.com",
    "aiplatform.googleapis.com",
    "bigqueryconnection.googleapis.com",
    "cloudresourcemanager.googleapis.com",
    "serviceusage.googleapis.com",
    "cloudaicompanion.googleapis.com",
    "geminidataanalytics.googleapis.com",
    f"--project={PROJECT_ID}",
    ],
    check=True,
    capture_output=True,  # Capture standard output and error stream
    text=True,            # Decode the bytes to a Python string automatically
)

print("APIs enabled successfully.")


---
## 2. Create Cloud Storage Bucket & Provision Lakehouse Iceberg Catalog

The **Lakehouse Iceberg REST Catalog** serves as the serverless metadata layer for table schemas, partitions, and snapshots while storing physical Parquet data files in Cloud Storage.

Let's create:
1. The Cloud Storage bucket `gs://${LAKEHOUSE_BUCKET_NAME}`.
2. The Lakehouse Iceberg REST Catalog named `${BUCKET_NAME}`.
3. A namespace inside the catalog `${NAMESPACE_ID}`.
3. An additional bucket that serves as our data starting point `gs://{EXTERNAL_BUCKET_NAME}`.

Let's also grant permissions for the Lakehouse catalog to manage `${LAKEHOUSE_BUCKET_NAME}`. This catalog uses `vended-credentials`, which means end-users only need permission on the Lakehouse table itself, not the underlying Storage bucket.



In [ ]:

# 1. Create Cloud Storage Bucket
storage_client = storage.Client(project=PROJECT_ID)

for BUCKET_NAME in [LAKEHOUSE_BUCKET_NAME, EXTERNAL_BUCKET_NAME]:
    try:
        bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
        print(f"Created bucket: gs://{BUCKET_NAME} in {REGION}")
    except Exception as e:
        if "409" in str(e) or "Conflict" in str(e) or "already exists" in str(e):
            bucket = storage_client.bucket(BUCKET_NAME)
            print(f"Bucket gs://{BUCKET_NAME} already exists.")
        else:
            raise e

# 2. Create Lakehouse Iceberg REST Catalog
subprocess.run([
    "gcloud", "biglake", "iceberg", "catalogs", "create", CATALOG_NAME,
    "--catalog-type=lakehouse",
    f"--project={PROJECT_ID}",
    f"--default-location=gs://{LAKEHOUSE_BUCKET_NAME}",
    "--credential-mode=vended-credentials",
    f"--restricted-locations=gs://{EXTERNAL_BUCKET_NAME}"
])

# 3. Create the Lakehouse Namespace
subprocess.run([
    "gcloud", "biglake", "iceberg", "namespaces", "create", NAMESPACE_ID,
    f"--catalog={CATALOG_NAME}",
    f"--project={PROJECT_ID}",
    f"--properties=location=gs://{EXTERNAL_BUCKET_NAME}"
])

print("✅ Lakehouse Iceberg REST Catalog provisioned successfully.")

# 4. Give the Lakehouse permission on both buckets
result = subprocess.run(
        [
            "gcloud", "biglake", "iceberg", "catalogs", "describe", 
            LAKEHOUSE_BUCKET_NAME,
            f"--project={PROJECT_ID}",
            "--format=value(biglake-service-account)"
        ],
        capture_output=True,  # Capture standard output and error stream
        text=True,            # Decode the bytes to a Python string automatically
    )

CATALOG_SA = result.stdout.strip()

subprocess.run([
    "gcloud", "storage", "buckets", "add-iam-policy-binding", f"gs://{LAKEHOUSE_BUCKET_NAME}",
    f"--member=serviceAccount:{CATALOG_SA}",
    "--role=roles/storage.objectUser"
])

Run the following set-up code to load your extra Cloud Storage bucket with data.

In [ ]:
from google.cloud import bigquery

# Initialize the client (Uses Application Default Credentials by default)
client = bigquery.Client(project=PROJECT_ID)

# Construct the EXPORT DATA SQL query
export_query = f"""
    EXPORT DATA OPTIONS (
        uri = 'gs://{EXTERNAL_BUCKET_NAME}/new_york_taxi_trips/tlc_yellow_trips_2022/*',
        format = 'PARQUET',
        overwrite = true
    ) AS (
        SELECT 
            vendor_id,
            pickup_datetime,
            dropoff_datetime,
            passenger_count,
            trip_distance,
            fare_amount,
            tip_amount,
            tolls_amount,
            total_amount,
            pickup_location_id,
            dropoff_location_id
        FROM 
            `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2022`
        LIMIT 1000000
    );
"""

query_job = client.query(export_query)
query_job.result()

---
## 3. Ingest & Process Cloud Storage data with Managed Apache Spark serverless

We will use **Managed Service for Apache Spark**to:
1. Load the taxi data into memory.
2. Filter rows with incomplete values.
3. Perform feature engineering to make the data more suitable for downstream workloads.
3. Connect and write the processed DataFrame directly to our **Lakehouse Apache Iceberg REST Catalog** as `{CATALOG_NAME}.{NAMESPACE_ID}.{TABLE_NAME}`.


In [ ]:
from google.cloud.dataproc_spark_connect import DataprocSparkSession
from pyspark.sql.functions import col, hour, when, to_date, unix_timestamp, round

spark = (
    DataprocSparkSession.builder.projectId(PROJECT_ID)
    .config("spark.sql.defaultCatalog", CATALOG_NAME) \
    .config(f'spark.sql.catalog.{CATALOG_NAME}', 'org.apache.iceberg.spark.SparkCatalog') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.type', 'rest') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.uri', 'https://biglake.googleapis.com/iceberg/v1/restcatalog') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.warehouse', f'bl://projects/{PROJECT_ID}/catalogs/{LAKEHOUSE_BUCKET_NAME}') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.io-impl', 'org.apache.iceberg.gcp.gcs.GCSFileIO') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.header.x-goog-user-project', PROJECT_ID) \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.rest.auth.type', 'org.apache.iceberg.gcp.auth.GoogleAuthManager') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.header.X-Iceberg-Access-Delegation', 'vended-credentials') \
    .config(f'spark.sql.catalog.{CATALOG_NAME}.gcs.oauth2.refresh-credentials-endpoint', 'https://oauth2.googleapis.com/token') \
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
    .location("us-central1")
    .getOrCreate()
)

# 2. Extract: Read raw taxi subset from Cloud Storage
# (Matches the schema exported using the BigQuery Python SDK script)
input_path = f"gs://{EXTERNAL_BUCKET_NAME}/new_york_taxi_trips/tlc_yellow_trips_2022/*"
print(f"Loading raw data from: {input_path}")

df_raw = spark.read.option("header", "true").option("inferSchema", "true").parquet(input_path)

# 3. Transform: Clean up raw fields and filter out invalid rows
print("Cleaning data and filtering anomalies...")
df_cleaned = df_raw.filter(
    (col("passenger_count") > 0) &
    (col("trip_distance") > 0.0) &
    (col("fare_amount") > 0.0) &
    (col("pickup_datetime").isNotNull()) &
    (col("dropoff_datetime").isNotNull())
)

# 4. Feature Engineering
print("Engineering temporal and financial metrics...")

# Calculate trip duration in minutes
duration_secs = unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))
df_enriched = df_cleaned.withColumn("trip_duration_mins", round(duration_secs / 60, 2))

# Filter out trips with invalid timestamps (dropoff before pickup or > 3 hours duration)
df_enriched = df_enriched.filter((col("trip_duration_mins") > 0.1) & (col("trip_duration_mins") < 180.0))

# Calculate average trip speed in MPH
df_enriched = df_enriched.withColumn(
    "avg_speed_mph", 
    round(col("trip_distance") / (col("trip_duration_mins") / 60), 2)
)

# Calculate tip percentage based on the base fare
df_enriched = df_enriched.withColumn(
    "tip_percentage", 
    round((col("tip_amount") / col("fare_amount")) * 100, 2)
)

# Extract the date for Iceberg partitioning
df_enriched = df_enriched.withColumn("pickup_date", to_date(col("pickup_datetime")))

# Map trip pickup times into logical traffic periods
pickup_hour = hour(col("pickup_datetime"))
df_enriched = df_enriched.withColumn(
    "commute_window",
    when((pickup_hour >= 7) & (pickup_hour <= 10), "Morning Rush")
    .when((pickup_hour >= 16) & (pickup_hour <= 19), "Evening Rush")
    .when((pickup_hour >= 11) & (pickup_hour <= 15), "Midday")
    .otherwise("Off Peak / Night")
)

# 5. Load: Save as partitioned Apache Iceberg table
# Partition by 'pickup_date' to show off Iceberg's fast partition-skipping abilities!
target_table = f"{NAMESPACE_ID}.{TABLE_NAME}_demo"
print(f"Writing cleaned dataset to Iceberg table: {target_table}")

df_enriched.write \
    .format("iceberg") \
    .mode("overwrite") \
    .partitionBy("pickup_date") \
    .saveAsTable(target_table)


You can view the table in your [Lakehouse](https://console.cloud.google.com/biglake/metastore/catalogs/) catalog by clicking your catalog name and then your namespace!

---
## 4. Query the Lakehouse Iceberg Table from BigQuery

Once the Spark job writes to the Iceberg REST Catalog, the table is automatically registered and immediately queryable across BigQuery!

The fully qualified table format is:
`` `{PROJECT_ID}.{LAKEHOUSE_BUCKET_NAME}.{NAMESPACE_ID}.{TABLE_NAME}` ``



In [ ]:
bq_client = bigquery.Client(project=PROJECT_ID, location=REGION)

query_lakehouse_sql = f"""
SELECT
  commute_window,
  avg(tip_percentage) as average_tip_percentage
FROM
  `{PROJECT_ID}.{LAKEHOUSE_BUCKET_NAME}.{NAMESPACE_ID}.{TABLE_NAME}`
GROUP BY commute_window
"""

for row in bq_client.query(query_lakehouse_sql).result():
  print(f"{row.commute_window}: ${row.average_tip_percentage:.2f}")


---
## 5. Generate Data Insights & Metadata with Knowledge Catalog

**Knowledge Catalog Data Insights** uses AI and Gemini to profile data distributions, generate natural language table summaries, document column schemas, and suggest analytical questions.

1. Navigate to [**BigQuery Studio**](https://console.cloud.google.com/bigquery) in Google Cloud Console.
2. In the Explorer pane, expand your project and select `{LAKEHOUSE_BUCKET_NAME}.{NAMESPACE_ID}.{TABLE_NAME}`.
3. Click on the **Insights** tab and click **Generate and publish**.
4. Review the AI-generated natural language table description and column descriptions.




---
## 6. Clean Up Resources (Optional)

Run this cell to remove all created resources and avoid ongoing Google Cloud charges.



In [ ]:
# UNCOMMENT AND RUN TO DELETE CREATED RESOURCES

# print(f"Deleting Lakehouse Iceberg table, namespace, and catalog...")
# !gcloud biglake iceberg tables delete {TABLE_NAME} --catalog={BUCKET_NAME} --namespace={NAMESPACE_ID} --quiet || true
# !gcloud biglake iceberg namespaces delete {NAMESPACE_ID} --catalog={BUCKET_NAME} --quiet || true
# !gcloud biglake iceberg catalogs delete {BUCKET_NAME} --quiet || true

# print(f"Deleting Knowledge Catalog taxonomy...")
# !gcloud data-catalog taxonomies delete {taxonomy_full_name} --quiet || true

# print(f"Deleting Cloud Storage Bucket: gs://{BUCKET_NAME}...")
# !gcloud storage rm -r gs://{BUCKET_NAME} || true

# print("✅ Cleanup complete.")
